# **USING OBJECT DETECTION TO DETECT PEST**

## **Set up**

In [1]:
import torch
import cv2
import albumentations as A
import zipfile
from ultralytics import YOLO

import sys
from pathlib import Path

TRAINING_ROOT = Path().resolve().parent
sys.path.append(str(TRAINING_ROOT))

from config.config import (
    DATA_DIR, CONFIG_DIR, MODEL_DIR,
    DATASET_ZIP, DATASET_DIR,
    TRAIN_CONFIG, N_AUGMENT, CLASS_NAMES
)
from utils.util import unzip_files, verify_dataset

CONFIG = CONFIG_DIR / 'data.yaml'
OUTPUT = MODEL_DIR  / 'runs'
OUTPUT.mkdir(parents=True, exist_ok=True)

## **Check GPU**

In [2]:
print(f'CUDA Available  : {torch.cuda.is_available()}')
print(f'CUDA Count      : {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'Device {i}     : {torch.cuda.get_device_name(i)}')

if torch.cuda.is_available():
    DEVICE = 0
    print(f'\nUsing GPU: {torch.cuda.get_device_name(DEVICE)}')
else:
    DEVICE = 'cpu'
    print('\nWARNING!!! CUDA not available - falling back to CPU.')

CUDA Available  : True
CUDA Count      : 1
Device 0     : NVIDIA GeForce RTX 5060 Laptop GPU

Using GPU: NVIDIA GeForce RTX 5060 Laptop GPU


## **Unzip Dataset**

In [3]:
unzip_files(DATASET_ZIP, DATASET_DIR)
verify_dataset(DATASET_DIR)

C:\Users\ASUS\Downloads\pest-detection\pest_detection_training\data\dataset already exists. Skipping unzip.
Verifying...
OK train --> images:  2610 | labels:  2556
OK val   --> images:   340 | labels:   316
OK test  --> images:   170 | labels:   157

Dataset is valid.


## **Augmentation**

In [4]:
augment_pipeline = A.Compose([
    # Geometry
    A.HorizontalFlip(p=0.5),
    A.Perspective(scale=(0.05, 0.1), p=0.3),
    # Color/Lighting
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.6),
    A.HueSaturationValue(hue_shift_limit=5, sat_shift_limit=30, val_shift_limit=20, p=0.3),
    A.CLAHE(clip_limit=2.0, p=0.3),
    A.RandomShadow(shadow_roi=(0, 0.5, 1, 1), p=0.2),
    # Noise/Blur
    A.GaussNoise(std_range=(0.01, 0.05), p=0.5),
    A.GaussianBlur(blur_limit=3, p=0.3),
    # Regularization
    A.CoarseDropout(
        num_holes_range=(2, 6),
        hole_height_range=(10, 40),
        hole_width_range=(10, 40),
        p=0.3
    ),
], bbox_params=A.BboxParams(
    format='yolo',
    label_fields=['class_labels'],
    min_visibility=0.3,
    clip=True
))


def read_label(path):
    boxes, labels = [], []
    if not Path(path).exists():
        return boxes, labels
    with open(path, 'r') as f:
        for line in f:
            parts = list(map(float, line.strip().split()))
            if len(parts) == 5:
                cls, x, y, w, h = parts
                x = max(0.0, min(1.0, x))
                y = max(0.0, min(1.0, y))
                w = max(0.0, min(1.0, w))
                h = max(0.0, min(1.0, h))
                boxes.append([x, y, w, h])
                labels.append(int(cls))
    return boxes, labels


def save_label(path, boxes, labels):
    with open(path, 'w') as f:
        for box, cls in zip(boxes, labels):
            f.write(f'{cls} {box[0]:.6f} {box[1]:.6f} {box[2]:.6f} {box[3]:.6f}\n')


def augment_dataset(img_dir, lbl_dir, n_augment=N_AUGMENT):
    img_dir = Path(img_dir)
    lbl_dir = Path(lbl_dir)

    # Remove old augmented files
    old_files = list(img_dir.glob('*_aug*.jpg')) + list(lbl_dir.glob('*_aug*.txt'))
    for f in old_files:
        f.unlink()
    print(f'Removed {len(old_files)} old augmented files.')

    img_files = list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png'))
    print(f'Augmenting {n_augment}x on {len(img_files)} images…')

    created = 0
    for img_path in img_files:
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        boxes, labels = read_label(str(lbl_dir / (img_path.stem + '.txt')))
        if not boxes:
            continue
        for i in range(n_augment):
            try:
                result = augment_pipeline(image=img, bboxes=boxes, class_labels=labels)
                if not result['bboxes']:
                    continue
                save_name = f'{img_path.stem}_aug{i}'
                cv2.imwrite(str(img_dir / f'{save_name}.jpg'), result['image'])
                save_label(str(lbl_dir / f'{save_name}.txt'), result['bboxes'], result['class_labels'])
                created += 1
            except Exception as e:
                print(f'Error {img_path.name}: {e}')

    orig = len(img_files)
    print(f'DONE! Original: {orig} | Added: {created} | Total: {orig + created}')


augment_dataset(
    img_dir=DATASET_DIR / 'images' / 'train',
    lbl_dir=DATASET_DIR / 'labels' / 'train',
)

Removed 3820 old augmented files.
Augmenting 3x on 700 images…
DONE! Original: 700 | Added: 1899 | Total: 2599


## **Train YOLOv8**

In [5]:
model = YOLO(TRAIN_CONFIG['model'])

results = model.train(
    data          = str(CONFIG),
    project       = str(OUTPUT),
    name          = 'run1',
    device        = DEVICE,
    epochs        = TRAIN_CONFIG['epochs'],
    imgsz         = TRAIN_CONFIG['imgsz'],
    batch         = TRAIN_CONFIG['batch'],
    workers       = TRAIN_CONFIG['workers'],
    patience      = TRAIN_CONFIG['patience'],
    optimizer     = TRAIN_CONFIG['optimizer'],
    lr0           = TRAIN_CONFIG['lr0'],
    lrf           = TRAIN_CONFIG['lrf'],
    weight_decay  = TRAIN_CONFIG['weight_decay'],
    cos_lr        = TRAIN_CONFIG['cos_lr'],
    warmup_epochs = TRAIN_CONFIG['warmup_epochs'],
    augment       = TRAIN_CONFIG['augment'],
    cache         = TRAIN_CONFIG['cache'],
    save          = TRAIN_CONFIG['save'],
    plots         = TRAIN_CONFIG['plots'],
    verbose       = TRAIN_CONFIG['verbose'],
    exist_ok      = TRAIN_CONFIG['exist_ok'],
)

BEST_MODEL_PATH = Path(results.save_dir) / 'weights' / 'best.pt'
print(f'\nBest model saved at: {BEST_MODEL_PATH}')

Ultralytics 8.4.51  Python-3.12.7 torch-2.12.0.dev20260408+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:\Users\ASUS\Downloads\pest-detection\pest_detection_training\config\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=run1, nbs=64, nms=Fal

## **Resume Training**

In [6]:
# model = YOLO(str(OUTPUT / 'run1/weights/last.pt'))
# results = model.train(resume=True)
# BEST_MODEL_PATH = Path(results.save_dir) / 'weights' / 'best.pt'

## **Evaluate on Validation**

In [ ]:
best_model = YOLO(str(BEST_MODEL_PATH))
v_metrics  = best_model.val(data=str(CONFIG), split='val', device=DEVICE)

print("===== RESULT ON VALIDATION =====")
print(f"mAP@0.5     : {v_metrics.box.map50:.4f}  ({v_metrics.box.map50*100:.1f}%)")
print(f"mAP@0.5:0.95: {v_metrics.box.map:.4f}  ({v_metrics.box.map*100:.1f}%)")
print(f"Precision   : {v_metrics.box.p.mean():.4f}")
print(f"Recall      : {v_metrics.box.r.mean():.4f}")
print()
for i, c in enumerate(CLASS_NAMES):
    try:
        ap = v_metrics.box.ap[i]
        print(f"{c:12s}: AP@0.5:0.95 = {ap:.4f} ({ap*100:.1f}%)")
    except:
        pass

Ultralytics 8.4.51  Python-3.12.7 torch-2.12.0.dev20260408+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8151MiB)
Model summary (fused): 73 layers, 11,126,745 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 897.0232.3 MB/s, size: 81.8 KB)
val: Scanning C:\Users\ASUS\Downloads\pest-detection\pest_detection_training\data\dataset\labels\val.cache... 316 images, 25 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 340/340  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 22/22 7.1it/s 3.1s0.1s
                   all        340       2963      0.747      0.756      0.789      0.472
               bo thon         28         32      0.733      0.906      0.918      0.687
                bo map        260       2028      0.791      0.805      0.825      0.394
                 bo to        238        903      0.717      0.558      0.623      0.333
Speed: 1.7ms preprocess, 4.0ms inference, 0.0ms loss

## **Evaluate on Test**

In [ ]:
t_metrics = best_model.val(data=str(CONFIG), split='test', device=DEVICE)

print('===== RESULT ON TEST =====')
print(f"mAP@0.5     : {t_metrics.box.map50:.4f}  ({t_metrics.box.map50*100:.1f}%)")
print(f"mAP@0.5:0.95: {t_metrics.box.map:.4f}  ({t_metrics.box.map*100:.1f}%)")
print(f"Precision   : {t_metrics.box.p.mean():.4f}")
print(f"Recall      : {t_metrics.box.r.mean():.4f}")
print()
for i, c in enumerate(CLASS_NAMES):
    try:
        ap = t_metrics.box.ap[i]
        print(f'{c:12s}: AP@0.5:0.95 = {ap:.4f} ({ap*100:.1f}%)')
    except:
        pass

Ultralytics 8.4.51  Python-3.12.7 torch-2.12.0.dev20260408+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8151MiB)
val: Fast image access  (ping: 0.00.0 ms, read: 132.439.6 MB/s, size: 66.2 KB)
val: Scanning C:\Users\ASUS\Downloads\pest-detection\pest_detection_training\data\dataset\labels\test... 157 images, 14 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 170/170 3.0Kit/s 0.1s
val: New cache created: C:\Users\ASUS\Downloads\pest-detection\pest_detection_training\data\dataset\labels\test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 11/11 6.4it/s 1.7s0.1s
                   all        170       1700      0.766      0.808      0.816      0.445
               bo thon         14         18      0.805      0.944      0.891      0.519
                bo map        127        926       0.81      0.843      0.891      0.497
                 bo to        104        756      0.682      0.638      0.667      0.319
Speed: 1.5ms p

## **Zip Results**

In [9]:
run1_dir = Path(results.save_dir)
zip_path = run1_dir.parent / 'run1.zip'

print(f'Zipping {run1_dir} …')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for file in run1_dir.rglob('*'):
        if file.is_file():
            zf.write(file, file.relative_to(run1_dir.parent))

print(f'Done! Saved to: {zip_path}')
print(f'File size: {zip_path.stat().st_size / 1024 / 1024:.1f} MB')

Zipping C:\Users\ASUS\Downloads\pest-detection\pest_detection_training\models\runs\run1 …
Done! Saved to: C:\Users\ASUS\Downloads\pest-detection\pest_detection_training\models\runs\run1.zip
File size: 45.2 MB
